
# Inteligência Computacional
## Estudo de Caso — Pré-processamento de Dados para Machine Learning

**Objetivo:** aplicar, em um problema realista de Ciência de Dados, as principais etapas de preparação de dados antes da modelagem.

### Cenário
Uma empresa de serviços por assinatura deseja prever quais clientes têm maior probabilidade de **cancelar o serviço** (`cancelou = 1`).

A base contém problemas comuns encontrados em projetos reais:

- valores ausentes;
- registros duplicados;
- categorias inconsistentes;
- tipos de dados incorretos;
- outliers;
- variáveis categóricas;
- atributos numéricos em escalas muito diferentes.

> A proposta não é apenas "limpar a base", mas justificar **por que** cada transformação é necessária e qual seu impacto potencial na modelagem.


## 1. Preparação do ambiente

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option("display.max_columns", None)
np.random.seed(42)



## 2. Construção do conjunto de dados

Para tornar o notebook autocontido, vamos gerar uma base sintética que reproduz problemas frequentes em dados reais.

A variável-alvo será **`cancelou`**:

- `0` → cliente permaneceu;
- `1` → cliente cancelou.


In [ ]:

n = 220

df = pd.DataFrame({
    "id_cliente": np.arange(1001, 1001+n),
    "idade": np.random.normal(38, 11, n).round(),
    "renda_mensal": np.random.normal(5200, 1900, n).round(2),
    "tempo_cliente_meses": np.random.randint(1, 72, n),
    "plano": np.random.choice(["Básico", "Intermediário", "Premium"], n, p=[0.45, 0.35, 0.20]),
    "cidade": np.random.choice(["Jundiaí", "Campinas", "São Paulo"], n, p=[0.35, 0.30, 0.35]),
    "chamados_suporte": np.random.poisson(2.2, n),
})

# Probabilidade de cancelamento: maior para pouco tempo de cliente,
# muitos chamados e plano básico.
score = (
    -0.03 * df["tempo_cliente_meses"]
    + 0.28 * df["chamados_suporte"]
    + 0.55 * (df["plano"] == "Básico").astype(int)
    - 0.00008 * df["renda_mensal"]
    + 0.35
)
prob = 1 / (1 + np.exp(-score))
df["cancelou"] = (np.random.rand(n) < prob).astype(int)

# Introdução proposital de problemas
df.loc[np.random.choice(df.index, 12, replace=False), "idade"] = np.nan
df.loc[np.random.choice(df.index, 10, replace=False), "renda_mensal"] = np.nan

# Categorias inconsistentes
df.loc[np.random.choice(df.index, 8, replace=False), "cidade"] = "são paulo"
df.loc[np.random.choice(df.index, 6, replace=False), "cidade"] = " SP "
df.loc[np.random.choice(df.index, 5, replace=False), "plano"] = "premium"

# Outliers
df.loc[np.random.choice(df.index, 3, replace=False), "renda_mensal"] = [28000, 35000, 42000]
df.loc[np.random.choice(df.index, 2, replace=False), "idade"] = [7, 112]

# Duplicidades
df = pd.concat([df, df.iloc[[4, 17, 44]]], ignore_index=True)

df.head()



## 3. Diagnóstico inicial

Antes de tratar, o cientista de dados deve **medir o problema**.

Perguntas orientadoras:

1. Quantas linhas e colunas existem?
2. Quais são os tipos de dados?
3. Há valores ausentes?
4. Existem registros duplicados?
5. As categorias estão padronizadas?
6. Existem valores extremos?


## 4. Valores ausentes — conceito e tratamento


### Decisão

- **Idade**: imputação pela **mediana**, por ser robusta a valores extremos.
- **Renda mensal**: também usaremos a **mediana**.
- Em projetos reais, a imputação deve considerar o mecanismo da ausência e o contexto de negócio.

> Remover linhas automaticamente nem sempre é a melhor decisão.


## 5. Duplicidades e inconsistências


### Padronização categórica

O mesmo conceito pode aparecer escrito de formas diferentes, criando categorias artificiais.

Exemplo:

`"São Paulo"`, `"são paulo"` e `" SP "` representam a mesma cidade.

O tratamento precisa ser **determinístico e documentado**.


## 6. Outliers: detectar não significa remover


### Decisão sobre outliers

Neste estudo, **não removeremos automaticamente** os outliers de renda.

Justificativa:

- renda elevada pode ser um valor legítimo;
- remover observações raras pode eliminar informação relevante;
- a decisão deve considerar contexto, origem do dado e algoritmo utilizado.

Para **idade**, valores biologicamente ou operacionalmente implausíveis podem indicar erro de coleta.



## 7. Separação entre atributos e variável-alvo

A variável `id_cliente` é apenas um identificador. Em geral, ela **não deve ser usada como variável preditora**, pois não representa uma característica do fenômeno que queremos modelar.



## 8. Divisão treino e teste **antes** de aprender transformações

Esta etapa evita **data leakage**.

As estatísticas usadas na imputação, codificação ou escalonamento devem ser aprendidas **somente no conjunto de treinamento**.


## 9. Codificação e escalonamento


### Variáveis numéricas
Serão:

1. imputadas pela mediana;
2. padronizadas com **Z-score**.

Formalmente:

\[
z = \frac{x - \mu}{\sigma}
\]

### Variáveis categóricas
Serão:

1. imputadas pela moda;
2. transformadas com **One-Hot Encoding**.

A transformação será encapsulada em um `ColumnTransformer`, permitindo que o processo seja reproduzível.



## 10. Pipeline: pré-processamento + modelo

Uma prática recomendada é conectar preparação e modelagem em um único objeto.

Isso reduz:

- divergências entre treino e teste;
- esquecimento de etapas;
- risco de vazamento de dados;
- dificuldade de reprodução do experimento.



## 11. O que foi feito?

| Problema | Diagnóstico | Tratamento |
|---|---|---|
| Valores ausentes | `isna()` | imputação |
| Duplicidades | `duplicated()` | remoção |
| Categorias inconsistentes | `value_counts()` | padronização |
| Tipos / valores inválidos | inspeção e regras | conversão/correção |
| Outliers | boxplot + IQR | análise contextual |
| Variáveis categóricas | tipo `object` | One-Hot Encoding |
| Escalas diferentes | estatísticas descritivas | padronização |
| Data leakage | desenho do experimento | divisão antes do `fit` |



## 12. Discussão

Responda em grupo:

1. Por que não removemos automaticamente todos os outliers?
2. Quando a média seria melhor que a mediana para imputação?
3. Por que `id_cliente` foi removido das variáveis preditoras?
4. O que aconteceria se ajustássemos o `StandardScaler` usando toda a base antes da divisão treino/teste?
5. Em quais algoritmos o escalonamento tende a ser mais importante?
6. O tratamento feito aqui serviria exatamente igual para uma árvore de decisão? Justifique.



## 13. Desafio

Modifique o estudo de caso:

- crie uma nova variável com valores ausentes;
- introduza uma nova inconsistência categórica;
- aplique uma estratégia diferente de imputação;
- compare o resultado do modelo;
- registre a justificativa para cada decisão.

> **Próxima etapa da disciplina:** Engenharia de Atributos e preparação orientada à modelagem.
